# Final Assignment: Analyzing Historical Stock/Revenue Data and Building a Dashboard

This notebook completes the six exercises for an IBM-style stock and revenue analysis assignment. It downloads historical stock prices with `yfinance`, collects quarterly revenue from Macrotrends using `requests` and BeautifulSoup, cleans the results with pandas, and creates interactive Plotly dashboards.

**Note:** Market prices and web-page tables change over time. Re-running the notebook refreshes the data and may produce different row counts or values.

## Setup

Install the required packages once if they are not already available:

```python
%pip install yfinance requests beautifulsoup4 lxml pandas plotly
```

In [1]:
import warnings

import pandas as pd
import requests
from bs4 import BeautifulSoup
import yfinance as yf
import plotly.graph_objects as go
from plotly.subplots import make_subplots

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

print('Libraries imported successfully.')

Libraries imported successfully.


In [2]:
def make_graph(stock_data, revenue_data, stock_name):
    """Create a two-panel stock-price and quarterly-revenue dashboard."""
    figure = make_subplots(rows=2, cols=1, shared_xaxes=True,
        subplot_titles=(f'{stock_name} Historical Share Price', f'{stock_name} Quarterly Revenue'), vertical_spacing=0.30)
    figure.add_trace(go.Scatter(x=stock_data['Date'], y=stock_data['Close'], name='Close price'), row=1, col=1)
    figure.add_trace(go.Scatter(x=revenue_data['Date'], y=revenue_data['Revenue'], name='Revenue'), row=2, col=1)
    figure.update_xaxes(title_text='Date', row=2, col=1)
    figure.update_yaxes(title_text='Price (USD)', row=1, col=1)
    figure.update_yaxes(title_text='Revenue (USD millions)', row=2, col=1)
    figure.update_layout(title=f'{stock_name}: Stock Price and Revenue', height=700, showlegend=False, template='plotly_white')
    return figure


def scrape_revenue(company):
    """Scrape a quarterly revenue table from IBM's course data pages."""
    base_url = ('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/'
                'IBMDeveloperSkillsNetwork-PY0220EN-SkillsNetwork/labs/project/')
    page = 'revenue.htm' if company == 'Tesla' else 'stock.html'
    response = requests.get(base_url + page, timeout=30)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, 'html.parser')
    for table in soup.find_all('table'):
        if f'{company} Quarterly Revenue' not in table.get_text(' ', strip=True):
            continue
        rows = []
        for row in table.find_all('tr'):
            values = [cell.get_text(strip=True) for cell in row.find_all(['th', 'td'])]
            if len(values) >= 2 and values[0] != 'Date':
                rows.append(values[:2])
        revenue = pd.DataFrame(rows, columns=['Date', 'Revenue'])
        revenue['Revenue'] = revenue['Revenue'].str.replace(r'[$,]', '', regex=True).replace('', pd.NA)
        revenue = revenue.dropna()
        revenue['Revenue'] = pd.to_numeric(revenue['Revenue'], errors='coerce')
        return revenue.dropna().reset_index(drop=True)
    raise ValueError(f'{company} quarterly revenue table was not found.')

## Exercise 1 — Extract Tesla stock data

Use `yfinance` to create a Tesla ticker object and obtain its complete price history.

In [3]:
tesla = yf.Ticker('TSLA')
tesla_data = tesla.history(period='max').reset_index()
tesla_data['Date'] = pd.to_datetime(tesla_data['Date']).dt.tz_localize(None)

print(f'Tesla stock rows: {len(tesla_data):,}')
tesla_data.head()

Tesla stock rows: 4,039


,Date,Open,High,Low,Close,Volume,Dividends,Stock Splits
0,2010-06-29,1.266667,1.666667,1.169333,1.592667,281494500,0.0,0.0
1,2010-06-30,1.719333,2.028000,1.553333,1.588667,257806500,0.0,0.0
2,2010-07-01,1.666667,1.728000,1.351333,1.464000,123282000,0.0,0.0
3,2010-07-02,1.533333,1.540000,1.247333,1.280000,77097000,0.0,0.0
4,2010-07-06,1.333333,1.333333,1.055333,1.074000,103003500,0.0,0.0


## Exercise 2 — Scrape Tesla revenue data

Retrieve Tesla's quarterly revenue table, remove currency formatting, and convert revenue to numeric values.

In [4]:
tesla_revenue = scrape_revenue('Tesla')

print(f'Tesla quarterly revenue rows: {len(tesla_revenue)}')
tesla_revenue.tail()

Tesla quarterly revenue rows: 53


,Date,Revenue
48,2010-09-30,31
49,2010-06-30,28
50,2010-03-31,21
51,2009-09-30,46
52,2009-06-30,27


## Exercise 3 — Extract GameStop stock data

Use `yfinance` to create a GameStop ticker object and obtain its complete price history.

In [5]:
gme = yf.Ticker('GME')
gme_data = gme.history(period='max').reset_index()
gme_data['Date'] = pd.to_datetime(gme_data['Date']).dt.tz_localize(None)

print(f'GameStop stock rows: {len(gme_data):,}')
gme_data.head()

GameStop stock rows: 6,147


,Date,Open,High,Low,Close,Volume,Dividends,Stock Splits
0,2002-02-13,1.620128,1.693350,1.603296,1.691666,76216000,0.0,0.0
1,2002-02-14,1.712707,1.716073,1.670626,1.683250,11021600,0.0,0.0
2,2002-02-15,1.683251,1.687459,1.658002,1.674834,8389600,0.0,0.0
3,2002-02-19,1.666418,1.666418,1.578047,1.607504,7410400,0.0,0.0
4,2002-02-20,1.615921,1.662210,1.603296,1.662210,6892800,0.0,0.0


## Exercise 4 — Scrape GameStop revenue data

Retrieve GameStop's quarterly revenue table, remove currency formatting, and convert revenue to numeric values.

In [6]:
gme_revenue = scrape_revenue('GameStop')

print(f'GameStop quarterly revenue rows: {len(gme_revenue)}')
gme_revenue.tail()

GameStop quarterly revenue rows: 62


,Date,Revenue
57,2006-01-31,1667
58,2005-10-31,534
59,2005-07-31,416
60,2005-04-30,475
61,2005-01-31,709


## Exercise 5 — Tesla dashboard

Create an interactive dashboard that compares Tesla's historical closing price with quarterly revenue.

In [7]:
tesla_figure = make_graph(tesla_data, tesla_revenue, 'Tesla')
tesla_figure.show()

## Exercise 6 — GameStop dashboard

Create an interactive dashboard that compares GameStop's historical closing price with quarterly revenue.

In [8]:
gme_figure = make_graph(gme_data, gme_revenue, 'GameStop')
gme_figure.show()

## Conclusion

The dashboards make it easier to explore each company’s market-price history alongside its reported quarterly revenue. The extraction and cleaning steps are deliberately reusable: changing the ticker and source URL lets the same workflow analyze another company.